# Lab 1 · 줄의 처짐과 하중

줄 위의 위치를 $x$, 수직 변위를 $u$, 하중을 나타내는 항을 $f$라 두면 세 문제 모두 $u''(x)=f(x)$로 쓸 수 있습니다.

| 실습 | 주어진 값 | 구할 값 |
| --- | --- | --- |
| [1.1 · 순방향 문제](#lab-11) | 길이 1, 일정한 하중, 양끝의 높이 | 위치별 변위 $u(x)$ |
| [1.2 · 길이가 달라지는 줄](#lab-12) | 줄의 길이 $L$, 일정한 하중, 양끝의 높이 | 길이에 따른 변위 $u(x,L)$ |
| [1.3 · 역문제](#lab-13) | 100개 지점의 높이, 양끝의 높이 | 변위 $u(x)$와 하중 $f(x)$ |

새 커널에서 1.2나 1.3부터 시작할 때는 먼저 1.1의 ‘1. 준비’에 있는 두 셀을 실행하세요.

<a id="lab-11"></a>

## Lab 1.1 · 양끝이 고정된 줄의 처짐

팽팽한 줄의 양끝을 같은 높이에 고정하고 줄 전체에 일정한 하중을 아래로 가합니다. 고정점의 높이를 0, 위쪽을 양수로 잡으면 줄이 처진 곳에서 변위는 음수입니다. 길이와 하중을 무차원화하면 식은 다음과 같습니다.

$$u''(x)=1,\qquad 0\leq x\leq 1,\qquad u(0)=u(1)=0.$$


### 두 곡선 중 어느 쪽이 줄의 모양일까요?

곡선 A는 $x(x-1)/2$, 곡선 B는 $-x(x-1)/2$입니다. 둘 다 양끝에서 0이 됩니다. 먼저 $u''=1$을 만족하는 곡선을 고른 뒤, 아래 셀에서 각 곡선의 양 끝값과 중점의 잔차 $u''-1$을 비교하세요.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("00_Start_Here에서 실습 커널과 GPU를 먼저 확인하세요.")

probe_x = torch.tensor([0.0, 0.5, 1.0], device="cuda", requires_grad=True)
for curve, sign in (("A", 1), ("B", -1)):
    probe_u = sign * probe_x * (probe_x - 1) / 2
    first_derivative, = torch.autograd.grad(probe_u.sum(), probe_x, create_graph=True)
    curvature, = torch.autograd.grad(first_derivative.sum(), probe_x)
    residual_at_middle = (curvature[1] - 1).item()
    print(f"{curve}    u(0) = {probe_u[0].item():z.0f},  u(1) = {probe_u[-1].item():z.0f},  "
          f"u″(0.5) − 1 = {residual_at_middle:.0f}")


### 1. 준비

A와 B의 양 끝값은 모두 0이지만, 중점에서 잔차는 각각 0과 −2입니다. 따라서 곡선 A가 방정식과 경계조건을 모두 만족합니다. 아래 두 셀을 실행해 학습에 쓸 함수와 경로를 준비하세요.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pinn_basics.py").is_file()
             and (p / "02_FNO.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("워크숍 교재 폴더 안에서 이 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("교재:", ROOT)
print("Python:", sys.executable)


In [ ]:
import json
import os
import subprocess
from uuid import uuid4
import numpy as np
import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("00_Start_Here의 GPU 확인부터 진행하세요.")
DEVICE = "cuda"
PINN_COMPLETED = False

LAB1_PROGRAM = ROOT / "01_labs/01_pinn/source_code/pinn_basics.py"
parameterized_run = None
inverse_run = None

def train_pinn_case(mode):
    output = ROOT / "runs" / f"pinn-{mode}-{uuid4().hex[:10]}"
    command = [sys.executable, str(LAB1_PROGRAM), "--mode", mode, "--device", DEVICE,
               "--steps", "1000", "--seed", "42", "--output-dir", str(output)]
    env = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
    subprocess.run(command, cwd=ROOT, env=env, check=True)
    print("결과:", output.relative_to(ROOT))
    return output


### 2. 순방향 PINN 학습

L-BFGS로 1,000회 학습합니다. 난수 시드는 42, 계산 정밀도는 FP32입니다. L-BFGS는 한 번의 업데이트에서 손실을 여러 차례 계산할 수 있습니다.


In [ ]:
PINN_COMPLETED = False
PINN_RUN = train_pinn_case("forward")
PINN_COMPLETED = True


### 3. 가운데 처짐과 오차

401개 위치에서 예측값을 해석해와 비교합니다. 아래 셀에 표시되는 세 값은 서로 다른 오차를 측정합니다.

- `solution_rmse` — 예측 변위와 해석해의 평균제곱근오차
- `pde_rmse` — 방정식 잔차의 실효값(RMS)
- `boundary_max_abs` — 양 끝점 중 절대 변위가 더 큰 쪽의 값


In [ ]:
if not PINN_COMPLETED:
    raise RuntimeError("바로 위 학습 셀을 완료하세요.")
with np.load(PINN_RUN / "predictions.npz", allow_pickle=False) as saved:
    x = saved["x"].reshape(-1)
    u_pred = saved["prediction"].reshape(-1)
    u_exact = saved["reference"].reshape(-1)
metrics = json.loads((PINN_RUN / "metrics.json").read_text())
print("배열 shape:", x.shape, u_pred.shape, u_exact.shape)
print(json.dumps(metrics["heldout_after"], indent=2))
middle = np.abs(x - 0.5).argmin()
print("가운데에서의 변위 (예측, 해석해):", u_pred[middle], u_exact[middle])


### 4. 예측값과 오차 그리기

직접 만들어 보려면 아래 요청문을 AI 코딩 도구에 붙여 넣고, 받은 코드로 다음 셀을 바꿔 실행하세요. [예제 코드](examples/pinn_plot.py)로 진행할 때는 셀을 그대로 실행하면 됩니다.

> 길이가 401인 NumPy 배열 `x`, `u_pred`, `u_exact`가 있습니다. 이 배열로 matplotlib 그래프 두 개를 나란히 그리는 코드를 작성하세요. 왼쪽은 위치 `x`에 따른 예측값과 해석해를 범례로 구분하고, 오른쪽은 `u_pred - u_exact`와 0 기준선을 표시하세요. 제목과 축 이름은 영어로 씁니다.


In [ ]:
from examples.pinn_plot import plot_pinn
from matplotlib import pyplot as plt

figure = plot_pinn(x, u_pred, u_exact)
display(figure)
plt.close(figure)


### 5. 가장 크게 처진 곳과 오차가 가장 큰 곳

해석해에서 줄이 가장 낮은 곳은 가운데이고 변위는 $-1/8$입니다. 예측값은 어디까지 내려갔나요? 오른쪽 그래프에서 0 기준선과 가장 멀리 떨어진 위치를 찾으세요. 양 끝점의 오차도 3절의 `boundary_max_abs`와 비교해 보세요.


<a id="lab-12"></a>

## Lab 1.2 · 줄의 길이가 달라지면

이번에는 오른쪽 고정점을 옮깁니다. 하중과 양끝의 높이는 그대로입니다. 신경망은 위치 $x$와 줄의 길이 $L$을 입력으로 받습니다.

$$\frac{\partial^2 u}{\partial x^2}=1,\qquad u(0,L)=u(L,L)=0,\qquad u(x,L)=\tfrac12 x(x-L).$$

길이 1에서 2 사이의 17개 길이에 대해 각각 내부 32개 지점에서 학습합니다. 아래 두 셀을 차례로 실행해 길이 1, 1.5, 2의 예측 곡선과 해석해를 겹쳐 보세요.


In [ ]:
parameterized_run = None
parameterized_run = train_pinn_case("parameterized")


In [ ]:
from matplotlib import pyplot as plt
from src.pinn_basics import BasicPINN, analytical

if parameterized_run is None:
    raise RuntimeError("Lab 1.2 학습 셀을 먼저 실행하세요.")
checkpoint = torch.load(parameterized_run / "model.pt", map_location="cpu", weights_only=True)
length_model = BasicPINN(checkpoint["config"], mode="parameterized", dtype=torch.float32)
length_model.load_state_dict(checkpoint["model_state_dict"])
length_model.eval()

figure, axis = plt.subplots(figsize=(7, 4))
print("길이 L     한가운데의 예측     한가운데의 해석해")
with torch.inference_mode():
    for length in (1.0, 1.5, 2.0):
        positions = torch.linspace(0.0, length, 401).reshape(-1, 1)
        reference = analytical(positions, length).flatten()
        predicted = length_model(positions, torch.full_like(positions, length)).flatten()
        line, = axis.plot(positions[:, 0], reference, label=f"L = {length:g} (exact)")
        axis.plot(positions[:, 0], predicted, "--", color=line.get_color(), label=f"L = {length:g} (PINN)")
        print(f"{length:5.2f}       {predicted[200].item(): .6f}             {reference[200].item(): .6f}")
axis.set(xlabel="Position x", ylabel="Displacement u", title="One PINN, three string lengths")
axis.legend(ncol=2)
figure.tight_layout()
display(figure)
plt.close(figure)


가운데 변위는 $u(L/2,L)=-L^2/8$입니다. 길이를 1에서 2로 늘리면 가운데 처짐은 0.125에서 0.5로 네 배가 됩니다. 그림에 나온 세 길이는 모두 모델을 학습할 때 사용한 길이입니다.


<a id="lab-13"></a>

## Lab 1.3 · 줄의 높이에서 하중 구하기

길이 1인 줄 위 100개 지점의 높이를 안다면, 하중이 어디에 얼마나 걸렸는지 추정할 수 있을까요? 이 예제의 관측값은 해석식으로 만든 값이어서 측정 잡음이 없습니다.

신경망 두 개를 함께 학습합니다. 하나는 관측한 높이에 맞춰 변위 $u(x)$를 예측하고, 다른 하나는 하중 $f(x)$를 추정합니다. 둘은 $u''(x)=f(x)$로 연결되며 양끝 변위는 0입니다. 기준 하중 $f(x)=x+\sin(4\pi x)$는 학습 뒤 추정값과 비교할 때 씁니다.

아래 두 셀을 실행하면 왼쪽에 변위, 오른쪽에 하중이 표시됩니다.


In [ ]:
inverse_run = None
inverse_run = train_pinn_case("inverse")


In [ ]:
from matplotlib import pyplot as plt

if inverse_run is None:
    raise RuntimeError("Lab 1.3 학습 셀을 먼저 실행하세요.")
with np.load(inverse_run / "predictions.npz", allow_pickle=False) as result:
    positions = result["x"].reshape(-1)
    figure, (height, load) = plt.subplots(1, 2, figsize=(10, 3.8))
    height.plot(positions, result["reference"], label="Exact displacement")
    height.plot(positions, result["prediction"], "--", label="PINN")
    load.plot(positions, result["source_reference"], label="True load")
    load.plot(positions, result["source_prediction"], "--", label="Inferred load")
height.set(xlabel="Position x", ylabel="Displacement u", title="String displacement")
load.set(xlabel="Position x", ylabel="Load f", title="Recovered load")
height.legend()
load.legend()
figure.tight_layout()
display(figure)
plt.close(figure)

errors = json.loads((inverse_run / "metrics.json").read_text())["heldout_after"]
print(f"변위 RMSE: {errors['solution_rmse']:.4g}")
print(f"하중 RMSE: {errors['source_rmse']:.4g}")
print(f"하중 최대 절대 오차: {errors['source_max_abs']:.4g}")


왼쪽에서 두 곡선이 겹쳐 보이더라도, 오른쪽에서 하중 추정값이 기준값과 얼마나 벌어지는지 확인하세요. 차이가 가장 큰 위치는 어디인가요?

[PINN 방정식과 학습 코드 살펴보기](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb)


[실습 목차](00_Start_Here.ipynb) · [다음 실습 · FNO로 열판의 온도 예측하기](02_FNO.ipynb)
